<a href="https://colab.research.google.com/github/salonapaliwal7/Machine-Learning-Projects/blob/main/Time_Series_Forecasting_using_ARIMA_%26_SARIMA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

store_sales_time_series_forecasting_path = kagglehub.competition_download('store-sales-time-series-forecasting')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import numpy as np
import pandas as pd

train_data = pd.read_csv("/kaggle/input/competitions/store-sales-time-series-forecasting/train.csv")
oil_data = pd.read_csv("/kaggle/input/competitions/store-sales-time-series-forecasting/oil.csv")
holidays_events = pd.read_csv("/kaggle/input/competitions/store-sales-time-series-forecasting/holidays_events.csv")
stores = pd.read_csv("/kaggle/input/competitions/store-sales-time-series-forecasting/stores.csv")
transactions = pd.read_csv("/kaggle/input/competitions/store-sales-time-series-forecasting/transactions.csv")
test_data = pd.read_csv("/kaggle/input/competitions/store-sales-time-series-forecasting/test.csv")

train_data.head()

# convert date to datetime
train_data['date'] = pd.to_datetime(train_data['date'])

In [ ]:
holidays_events.head()

# convert date to datetime
holidays_events['date'] = pd.to_datetime(holidays_events['date'])

In [ ]:
oil_data.head()

# convert date to datetime
oil_data['date'] = pd.to_datetime(oil_data['date'])

In [ ]:
stores.head()

In [ ]:
transactions.head()

# convert date to datetime
transactions['date'] = pd.to_datetime(transactions['date'])

In [ ]:
train_data['store_nbr'].nunique()

In [ ]:
train_data['family'].nunique()

In [ ]:
train_data['date'].min()

In [ ]:
train_data['date'].max()

In [ ]:
# check for null values
print("train data " , train_data.isnull().sum())
print("oil data " , oil_data.isnull().sum())
print("holidays events " , holidays_events.isnull().sum())
print("stores " , stores.isnull().sum())
print("transactions " , transactions.isnull().sum())
print("test data ", test_data.isnull().sum())

In [ ]:
# check for duplicates
print("train data " , train_data.duplicated().sum())
print("oil data " , oil_data.duplicated().sum())
print("holidays events " , holidays_events.duplicated().sum())
print("stores " , stores.duplicated().sum())
print("transactions " , transactions.duplicated().sum())
print("test data " , test_data.duplicated().sum())

In [ ]:
# check why oil prices were null
oil_data['day_name'] = oil_data['date'].dt.day_name()
oil_data[oil_data['dcoilwtico'].isnull()][['date', 'day_name']]

In [ ]:
# Oil prices are missing on market holidays.
# Use forward fill to propagate the last known price.
# Backfill handles the first missing observation where no previous value exists.
oil_data['dcoilwtico'] = oil_data['dcoilwtico'].ffill().bfill()

In [ ]:
# lets check if series is stationary or not

import matplotlib.pyplot as plt

daily_sales = train_data.groupby('date')['sales'].sum().reset_index()
# visualization
plt.figure(figsize=(15,6))
plt.plot(daily_sales['date'], daily_sales['sales'])
plt.title("Daily Total Sales")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.show()

In [ ]:
# Lets check dates for these sudden drops to 0 at year start mostly
daily_sales.sort_values('sales').head(10)

In [ ]:
# rolling mean

daily_sales['rolling_30'] = (
    daily_sales['sales']
    .rolling(window=30)
    .mean()
)

plt.figure(figsize=(15,6))

plt.plot(daily_sales['date'],
         daily_sales['sales'],
         alpha=0.4,
         label='Daily Sales')

plt.plot(daily_sales['date'],
         daily_sales['rolling_30'],
         color='red',
         linewidth=3,
         label='30-Day Rolling Mean')

plt.legend()
plt.show()

In [ ]:
# decompose

from statsmodels.tsa.seasonal import seasonal_decompose

decomposition = seasonal_decompose(
    daily_sales['sales'],
    model='additive',
    period=7   # weekly seasonality
)

decomposition.plot();

In [ ]:
# ADF tests

from statsmodels.tsa.stattools import adfuller
result = adfuller(daily_sales['sales'])
print("ADF Statistic:", result[0])
print("p-value:", result[1])

In [ ]:
# p-value > 0.05
# non-stationary

# Lets do first order differencing

daily_sales['sales_diff'] = daily_sales['sales'].diff()
plt.figure(figsize=(15,6))

plt.plot(daily_sales.index,
         daily_sales['sales_diff'])

plt.title("First-order Differenced Series")
plt.show()

In [ ]:
result = adfuller(
    daily_sales['sales_diff'].dropna()
)

print(result[1])

 Stationarity Check

- The original time series had an ADF p-value of **0.0897**, which is greater than 0.05.
- Hence, we failed to reject the null hypothesis and concluded that the series was **non-stationary**.
- First-order differencing was applied.
- After differencing, the ADF p-value became **4.57e-21**, which is much smaller than 0.05.
- Therefore, the differenced series is **stationary**.
- Hence, for ARIMA, the differencing order is **d = 1**.

In [ ]:
# to chose p and q

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, ax = plt.subplots(1, 2, figsize=(15,5))

plot_acf(
    daily_sales['sales_diff'].dropna(),
    lags=30,
    ax=ax[0]
)

plot_pacf(
    daily_sales['sales_diff'].dropna(),
    lags=30,
    ax=ax[1]
)

plt.show()

In [ ]:
# Lets build ARIMA model

train = daily_sales.iloc[:-30]
test = daily_sales.iloc[-30:]

from statsmodels.tsa.arima.model import ARIMA

model = ARIMA(
    train['sales'],
    order=(1,1,1)
)

model_fit = model.fit()

print(model_fit.summary())


In [ ]:
forecast = model_fit.forecast(steps=30)

In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
import numpy as np

mae = mean_absolute_error(test['sales'], forecast)

rmse = np.sqrt(
    mean_squared_error(test['sales'], forecast)
)

print(mae)
print(rmse)

In [ ]:
plt.figure(figsize=(15,6))

plt.plot(train.index,
         train['sales'],
         label='Train')

plt.plot(test.index,
         test['sales'],
         label='Actual')

plt.plot(test.index,
         forecast,
         label='ARIMA Forecast')

plt.legend()
plt.show()

In [ ]:
# from the plot we infer that ARIMA is unable to capture the weekly seasonal fluctuations lets try SARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

sarima = SARIMAX(
    train['sales'],
    order=(1,1,1),
    seasonal_order=(1,1,1,7) # m = 7 → Weekly seasonality (daily data)
)

sarima_fit = sarima.fit()

forecast = sarima_fit.forecast(steps=30)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(test['sales'], forecast)
rmse = np.sqrt(mean_squared_error(test['sales'], forecast))

print(mae)
print(rmse)

In [ ]:
plt.figure(figsize=(15,6))

plt.plot(train.index,
         train['sales'],
         label='Train')

plt.plot(test.index,
         test['sales'],
         label='Actual')

plt.plot(test.index,
         forecast,
         label='SARIMA Forecast')

plt.legend()
plt.show()

SARIMA outperformed ARIMA because the sales data exhibits a clear weekly seasonal pattern. ARIMA captures the overall trend after differencing but cannot explicitly model seasonality, whereas SARIMA incorporates seasonal autoregressive, differencing, and moving average components.